In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("../results/video_test/student_data1.csv")

df.head()

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

In [ ]:
missing_by_student = (
    df.groupby("student_id")
      .apply(lambda x: x.isna().sum())
)

missing_by_student

In [ ]:
features = [
    "yaw",
    "pitch",
    "roll",
    "avg_ear",
    "gaze",
    "shoulder_tilt",
    "head_drop",
    "head_offset",
    "hand_raised",
    "hands_near_face"
]

availability = (
    df.groupby("student_id")[features]
      .apply(lambda x: x.notna().mean() * 100)
      .round(2)
)

availability

In [ ]:
availability.plot(
    kind="bar",
    figsize=(14, 6)
)

plt.title("Feature Availability by Student")
plt.xlabel("Student ID")
plt.ylabel("Available Data (%)")
plt.ylim(0, 105)
plt.xticks(rotation=0)
plt.legend(
    title="Features",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
student_id = 3

student = df[df["student_id"] == student_id]

plt.figure(figsize=(14, 5))

plt.plot(student["time"], student["yaw"], label="Yaw")
plt.plot(student["time"], student["pitch"], label="Pitch")
plt.plot(student["time"], student["roll"], label="Roll")

plt.xlabel("Time (seconds)")
plt.ylabel("Angle (degrees)")
plt.title(f"Head Pose Over Time — Student {student_id}")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
student_id = 19

student = df[df["student_id"] == student_id]

plt.figure(figsize=(14, 5))

plt.plot(student["time"], student["left_ear"], label="Left EAR")
plt.plot(student["time"], student["right_ear"], label="Right EAR")
plt.plot(student["time"], student["avg_ear"], label="Average EAR")

# Every student has a different normal EAR, so the closure level is
# relative to this student's own 75th percentile (see src/eye_features.py).
plt.axhline(
    y=0.6 * student["avg_ear"].quantile(0.75),
    linestyle="--",
    label="Closure level (0.6 x own baseline)"
)

plt.xlabel("Time (seconds)")
plt.ylabel("EAR")
plt.title(f"Eye Aspect Ratio Over Time — Student {student_id}")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
student_id = 7

student = df[df["student_id"] == student_id].copy()

# The old "eye_state" column used a fixed EAR < 0.22 rule and was removed.
# Closed here = EAR below 0.6 x the student's own normal level.
ratio = student["avg_ear"] / student["avg_ear"].quantile(0.75)
eye_open = (ratio >= 0.6).astype(float).where(ratio.notna())

plt.figure(figsize=(14, 3))

plt.step(
    student["time"],
    eye_open,
    where="post"
)

plt.yticks([0, 1], ["Closed", "Open"])
plt.xlabel("Time (seconds)")
plt.ylabel("Eye State")
plt.title(f"Eyes Closed / Open Over Time — Student {student_id}")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
gaze_counts = (
    df.groupby(["student_id", "gaze"])
      .size()
      .unstack(fill_value=0)
)

gaze_percentage = (
    gaze_counts
    .div(gaze_counts.sum(axis=1), axis=0)
    .mul(100)
    .round(2)
)

gaze_percentage

In [ ]:
student_id = 7

student = df[df["student_id"] == student_id].copy()

gaze_mapping = {
    "DOWN": 0,
    "RIGHT": 1,
    "FORWARD": 2,
    "LEFT": 3
}

student["gaze_numeric"] = student["gaze"].map(gaze_mapping)

plt.figure(figsize=(14, 4))

plt.step(
    student["time"],
    student["gaze_numeric"],
    where="post"
)

plt.yticks(
    [0, 1, 2, 3],
    ["DOWN", "RIGHT", "FORWARD", "LEFT"]
)

plt.xlabel("Time (seconds)")
plt.ylabel("Gaze Direction")
plt.title(f"Gaze Direction Over Time — Student {student_id}")

plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
student_id = 3

student = df[df["student_id"] == student_id]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

ax1.plot(student["time"], student["shoulder_tilt"], label="Shoulder tilt (deg)")
ax1.set_ylabel("Angle (degrees)")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(student["time"], student["head_drop"], label="Head drop")
ax2.plot(student["time"], student["head_offset"], label="Head offset")
ax2.set_xlabel("Time (seconds)")
ax2.set_ylabel("Fraction of shoulder width")
ax2.legend()
ax2.grid(alpha=0.3)

ax1.set_title(f"Upper-body Posture Over Time — Student {student_id}")
plt.tight_layout()
plt.show()

In [ ]:
student_id = 3

student = df[df["student_id"] == student_id]

plt.figure(figsize=(14, 4))

plt.plot(
    student["time"],
    student["head_drop"],
    label="Head drop"
)

plt.xlabel("Time (seconds)")
plt.ylabel("Head drop (fraction of shoulder width)")
plt.title(f"Head Drop Over Time — Student {student_id}")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
student_frames = (
    df.groupby("student_id")["frame"]
      .nunique()
      .sort_index()
)

student_frames

In [ ]:
tracking_duration = (
    df.groupby("student_id")
      .agg(
          first_frame=("frame", "min"),
          last_frame=("frame", "max"),
          frames_seen=("frame", "nunique"),
          first_time=("time", "min"),
          last_time=("time", "max")
      )
)

tracking_duration["duration_sec"] = (
    tracking_duration["last_time"] - tracking_duration["first_time"]
)

tracking_duration

In [ ]:
plt.figure(figsize=(14, 6))

for student_id, group in df.groupby("student_id"):
    plt.scatter(
        group["time"],
        [student_id] * len(group),
        s=8
    )

plt.xlabel("Time (seconds)")
plt.ylabel("Student ID")
plt.title("Student Tracking Coverage Over Time")

plt.yticks(sorted(df["student_id"].unique()))
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
gaze_overall = (
    df["gaze"]
    .value_counts(dropna=True)
    .sort_values(ascending=False)
)

gaze_overall

In [ ]:
gaze_percentage = (
    df["gaze"]
    .value_counts(dropna=True)
    .div(df["gaze"].notna().sum())
    .mul(100)
    .round(2)
)

gaze_percentage

In [ ]:
# Share of frames with closed / open eyes per student (closed = EAR below
# 0.6 x the student's own normal level; frames without EAR are left out).
ratio = df["avg_ear"] / df.groupby("student_id")["avg_ear"].transform(
    lambda x: x.quantile(0.75)
)

eye_label = pd.Series(
    np.where(ratio < 0.6, "CLOSED", "Open"), index=df.index
).where(ratio.notna())

eye_distribution = (
    df.assign(eye_label=eye_label)
      .groupby(["student_id", "eye_label"])
      .size()
      .unstack(fill_value=0)
)

eye_percentage = (
    eye_distribution
    .div(eye_distribution.sum(axis=1), axis=0)
    .mul(100)
    .round(2)
)

eye_percentage

In [ ]:
student_id = 6

student = df[df["student_id"] == student_id]

plt.figure(figsize=(14, 5))

plt.plot(
    student["time"],
    student["avg_ear"],
    label="Average EAR"
)

plt.axhline(
    y=0.6 * student["avg_ear"].quantile(0.75),
    linestyle="--",
    label="Closure level (0.6 x own baseline)"
)

plt.xlabel("Time (seconds)")
plt.ylabel("EAR")
plt.title(f"EAR and Eye Closure — Student {student_id}")

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
student_id = 6

student = df[df["student_id"] == student_id].copy()

gaze_mapping = {
    "DOWN": 0,
    "RIGHT": 1,
    "FORWARD": 2,
    "LEFT": 3
}

student["gaze_numeric"] = student["gaze"].map(gaze_mapping)

fig, ax1 = plt.subplots(figsize=(14, 5))

# Gaze
ax1.step(
    student["time"],
    student["gaze_numeric"],
    where="post",
    label="Gaze"
)

ax1.set_yticks([0, 1, 2, 3])
ax1.set_yticklabels(["DOWN", "RIGHT", "FORWARD", "LEFT"])
ax1.set_xlabel("Time (seconds)")
ax1.set_ylabel("Gaze Direction")

# Eye state on second axis
ax2 = ax1.twinx()

eye_numeric = (
    (student["avg_ear"] / student["avg_ear"].quantile(0.75) >= 0.6)
    .astype(float)
    .where(student["avg_ear"].notna())
)

ax2.step(
    student["time"],
    eye_numeric,
    where="post",
    linestyle="--",
    label="Eye State"
)

ax2.set_yticks([0, 1])
ax2.set_yticklabels(["Closed", "Open"])
ax2.set_ylabel("Eye State")

plt.title(f"Gaze + Eyes Closed/Open Over Time — Student {student_id}")
plt.tight_layout()
plt.show()

In [ ]:
signal_availability = (
    df.groupby("student_id")
      .agg(
          head_pose_valid=("yaw", lambda x: x.notna().mean() * 100),
          eye_valid=("avg_ear", lambda x: x.notna().mean() * 100),
          gaze_valid=("gaze", lambda x: x.notna().mean() * 100),
          posture_valid=("head_drop", lambda x: x.notna().mean() * 100)
      )
      .round(2)
)

signal_availability